In [1]:
# Imports and settings
# 说明：初始化依赖与设备，准备后续训练。
import os  # 文件路径
import json  # 结果保存
import math  # 数学工具
import random  # 随机数
import numpy as np  # 数值计算
import pandas as pd  # 表格处理
import torch  # 深度学习框架
from sklearn.metrics import mean_squared_error  # 评估指标
from torch_geometric.data import Data  # 图数据结构
from torch_geometric.nn import RGCNConv  # 关系图卷积
import torch.nn.functional as F  # 激活函数

# Reproducibility and device
seed = 42  # 随机种子
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 计算设备
print('device ->', device)

device -> cuda


In [2]:
# Data loading and rolling feature utilities
# 说明：只使用历史收益构造特征，避免时间泄漏。
def load_returns_csv(path):
    """读取收益率矩阵。"""
    if not os.path.exists(path):
        raise FileNotFoundError('收益率文件不存在')
    df = pd.read_csv(path)
    date_col = 'trade_date' if 'trade_date' in df.columns else df.columns[0]
    dates = pd.to_datetime(df[date_col], errors='coerce')
    rets = df.drop(columns=[date_col], errors='ignore')
    rets.columns = [str(c) for c in rets.columns]
    rets = rets.apply(pd.to_numeric, errors='coerce')
    rets.index = dates
    return rets


def zscore_by_row(feat_df):
    """对每个特征按横截面标准化。"""
    mean = feat_df.mean(axis=0)
    std = feat_df.std(axis=0).replace(0, np.nan)
    return (feat_df - mean) / std


def build_features_from_window(window_df):
    """用窗口收益构造特征（不含未来信息）。"""
    mom20 = (1 + window_df.tail(20)).prod() - 1
    mean5 = window_df.tail(5).mean()
    mean10 = window_df.tail(10).mean()
    vol20 = window_df.tail(20).std(ddof=0)
    vol60 = window_df.tail(60).std(ddof=0)
    last_ret = window_df.tail(1).iloc[0]
    feat = pd.DataFrame({
        'mom20': mom20,
        'mean5': mean5,
        'mean10': mean10,
        'vol20': vol20,
        'vol60': vol60,
        'last_ret': last_ret
    })
    return zscore_by_row(feat)


def load_industry_mapping(path, stock_codes):
    """读取行业映射并对齐。"""
    if not os.path.exists(path):
        raise FileNotFoundError('行业映射文件不存在')
    df = pd.read_csv(path)
    df['ts_code'] = df['ts_code'].astype(str)
    df = df[df['ts_code'].isin(stock_codes)].copy()
    return df

In [ ]:
# Edge building utilities
# 说明：相关性边只使用历史窗口构建，行业边为静态边。
def build_stock2idx(stock_codes):
    return {code: idx for idx, code in enumerate(stock_codes)}


def edge_index_from_pairs(edge_pairs):
    if not edge_pairs:
        return torch.empty((2, 0), dtype=torch.long)
    pair_tensor = torch.tensor(sorted(edge_pairs), dtype=torch.long)
    return pair_tensor.t().contiguous()


def build_industry_edges(industry_df, stock2idx, max_neighbors=20):
    edge_pairs = set()
    grouped = industry_df.groupby('industry')['ts_code'].apply(list).to_dict()
    for members in grouped.values():
        members = [m for m in members if m in stock2idx]
        for src in members:
            peers = [m for m in members if m != src]
            if len(peers) > max_neighbors:
                peers = peers[:max_neighbors]
            for dst in peers:
                edge_pairs.add((stock2idx[src], stock2idx[dst]))
    return edge_index_from_pairs(edge_pairs)


def build_corr_edges(window_df, top_k=20):
    data = window_df.fillna(0.0).to_numpy(dtype=float)
    if data.shape[0] < 5:
        return torch.empty((2, 0), dtype=torch.long)
    corr = np.corrcoef(data.T)
    edge_pairs = set()
    for i in range(corr.shape[0]):
        row = corr[i].copy()
        row[i] = -np.inf
        idx = np.argsort(row)[-top_k:]
        for j in idx:
            if np.isfinite(row[j]):
                edge_pairs.add((i, j))
    return edge_index_from_pairs(edge_pairs)

In [ ]:
# Dataset construction
# 说明：按日期构建特征、边和标签，避免时间泄漏。
returns_path = 'data/daily_returns.csv'  # 收益率路径
industry_path = 'data/industry_mapping.csv'  # 行业映射路径
rets = load_returns_csv(returns_path)
stock_codes = list(rets.columns)  # 股票列表
stock2idx = build_stock2idx(stock_codes)
industry_df = load_industry_mapping(industry_path, stock_codes)

lookback = 60  # 回看窗口
top_k = 20  # 相关性邻居数
data_list = []  # 数据列表
date_list = []  # 日期列表
for t in range(lookback, len(rets) - 1):
    window_df = rets.iloc[t - lookback:t]
    feat_df = build_features_from_window(window_df)
    feat_df = feat_df.reindex(stock_codes).fillna(0.0)
    x = torch.tensor(feat_df.to_numpy(dtype=float), dtype=torch.float)
    y = torch.tensor(rets.iloc[t + 1].values, dtype=torch.float)
    mask = ~torch.isnan(y)
    ind_edge = build_industry_edges(industry_df, stock2idx, max_neighbors=20)
    corr_edge = build_corr_edges(window_df, top_k=top_k)
    edge_index = torch.cat([ind_edge, corr_edge], dim=1)
    edge_type = torch.cat([
        torch.zeros(ind_edge.size(1), dtype=torch.long),
        torch.ones(corr_edge.size(1), dtype=torch.long)
    ])
    data = Data(x=x, edge_index=edge_index, edge_type=edge_type, y=y, train_mask=mask, val_mask=mask)
    data_list.append(data)
    date_list.append(rets.index[t])
print('可用样本天数:', len(data_list))

train_window = 200  # 训练窗口
val_window = 50  # 验证窗口
step = 10  # 步长
total_days = len(data_list)
if total_days < train_window + val_window:
    train_window = max(1, total_days - val_window)
window_splits = [(list(range(s, s + train_window)), list(range(s + train_window, s + train_window + val_window)))
                 for s in range(0, total_days - train_window - val_window + 1, step)]
if len(window_splits) == 0:
    raise ValueError('滑动窗口数量为 0，请调整参数')
print('滑动窗口数量:', len(window_splits))

可用样本天数: 71
滑动窗口数量: 1


In [5]:
# Model and training utilities
# 说明：使用 MSE 训练，IC 只用于评估。
class StockRGCN_Regression(torch.nn.Module):
    """两层 RGCN 回归模型。"""
    def __init__(self, in_channels, hidden_channels=64, num_relations=2):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, hidden_channels, num_relations)
        self.regressor = torch.nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_type)
        return self.regressor(x).squeeze(-1)


def masked_mse(pred, target, mask):
    if mask.sum() == 0:
        return torch.tensor(float('nan'), device=pred.device)
    diff = pred[mask] - target[mask]
    return (diff * diff).mean()


def compute_ic_np(y_true, y_pred, mask):
    if mask.sum() < 2:
        return float('nan')
    try:
        return float(np.corrcoef(y_true[mask], y_pred[mask])[0, 1])
    except Exception:
        return float('nan')


def train_one_day(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data.x.to(device), data.edge_index.to(device), data.edge_type.to(device))
    loss = masked_mse(out, data.y.to(device), data.train_mask.to(device))
    if torch.isnan(loss):
        return float('nan')
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return float(loss.item())


def eval_one_day(model, data):
    model.eval()
    with torch.no_grad():
        out = model(data.x.to(device), data.edge_index.to(device), data.edge_type.to(device))
        pred = out.cpu().numpy()
        y = data.y.cpu().numpy()
        mask = data.val_mask.cpu().numpy()
        mse = mean_squared_error(y[mask], pred[mask]) if mask.sum() > 0 else float('nan')
        ic = compute_ic_np(y, pred, mask)
    return float(mse), float(ic)

In [ ]:
# Training loop
# 说明：严格按时间窗口训练与验证。
in_channels = data_list[0].x.size(1)  # 特征维度
hidden_channels = 64  # 隐层维度
learning_rate = 1e-3  # 学习率
epochs = 20  # 训练轮数
patience = 3  # 早停轮数

best_ic = -1e9
best_state = None
window_metrics = []

for window_id, (train_idx, val_idx) in enumerate(window_splits, start=1):
    print(f'窗口 {window_id}/{len(window_splits)} 开始训练')
    model = StockRGCN_Regression(in_channels=in_channels, hidden_channels=hidden_channels, num_relations=2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    bad_epochs = 0
    best_window_ic = -1e9
    best_window_state = None
    for epoch in range(1, epochs + 1):
        losses = []
        for t in train_idx:
            loss = train_one_day(model, data_list[t], optimizer)
            if not np.isnan(loss):
                losses.append(loss)
        val_mse_list = []
        val_ic_list = []
        for t in val_idx:
            mse, ic = eval_one_day(model, data_list[t])
            val_mse_list.append(mse)
            val_ic_list.append(ic)
        val_ic = float(np.nanmean(val_ic_list))
        if not np.isnan(val_ic) and val_ic > best_window_ic:
            best_window_ic = val_ic
            best_window_state = {k: v.cpu() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if bad_epochs >= patience:
            break
    window_metrics.append({'window_id': window_id, 'best_val_ic': best_window_ic})
    if best_window_ic > best_ic:
        best_ic = best_window_ic
        best_state = best_window_state
    print(f'窗口 {window_id} 完成，最佳 IC={best_window_ic:.6f}')

if best_state is not None:
    torch.save(best_state, 'best_model.pt')
    print('已保存 best_model.pt')

metrics = {'window_metrics': window_metrics, 'best_ic': best_ic}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f)
print('已保存 metrics.json')

窗口 1/1 开始训练


ValueError: Input contains NaN.